In [1]:
import json
from pathlib import Path

import numpy as np

from pyscf import gto

import sys

BASIS = "def2-svp"

In [2]:
import yaml
dict_ = {"molecule": [], "spin": {}, "charge": {}}
dataset = "s66x8"
dict_[f"molecule_{dataset}"] = []
dict_[f"reaction-{dataset}"] = {}

data_path = Path("./sets")

# read s66x8.yaml
class RubyTagSafeLoader(yaml.SafeLoader):
    pass


def _ignore_unknown_tags(loader, tag_suffix, node):
    if isinstance(node, yaml.MappingNode):
        return loader.construct_mapping(node, deep=True)
    if isinstance(node, yaml.SequenceNode):
        return loader.construct_sequence(node, deep=True)
    return loader.construct_scalar(node)


RubyTagSafeLoader.add_multi_constructor("!", _ignore_unknown_tags)

with open(data_path / "s66x8.yaml", "r") as f:
    s66x8 = yaml.load(f, Loader=RubyTagSafeLoader)

print(s66x8["items"])

[{'name': '01 Water ... Water 0.90', 'curve': 1, 'curve_name': 'Water ... Water', 'curve_x': 0.9, 'shortname': '01_Water-Water_0.90', 'geometry': 'S66x8:01_0.90', 'reference_value': -4.573, 'setup': {}, 'group': 'H-bonds', 'tags': '1 H-bond'}, {'name': '01 Water ... Water 0.95', 'curve': 1, 'curve_name': 'Water ... Water', 'curve_x': 0.95, 'shortname': '01_Water-Water_0.95', 'geometry': 'S66x8:01_0.95', 'reference_value': -4.884, 'setup': {}, 'group': 'H-bonds', 'tags': '1 H-bond'}, {'name': '01 Water ... Water 1.00', 'curve': 1, 'curve_name': 'Water ... Water', 'curve_x': 1.0, 'shortname': '01_Water-Water_1.00', 'geometry': 'S66x8:01_1.00', 'reference_value': -4.894, 'setup': {}, 'group': 'H-bonds', 'tags': '1 H-bond'}, {'name': '01 Water ... Water 1.05', 'curve': 1, 'curve_name': 'Water ... Water', 'curve_x': 1.05, 'shortname': '01_Water-Water_1.05', 'geometry': 'S66x8:01_1.05', 'reference_value': -4.723, 'setup': {}, 'group': 'H-bonds', 'tags': '1 H-bond'}, {'name': '01 Water ... Wa

In [3]:
curve_geom_dict = {}

for items in s66x8["items"]:
    xyz_file = (Path("sets/S66x8") / f"{items['shortname']}.xyz").resolve().as_posix()
    mol = gto.M(
        atom=xyz_file,
        basis=BASIS,
        verbose=0,
        charge=0,
        spin=0,
        unit="B",
    )
    mol.build()

    molecule = []
    for i_atom in mol._atom:
        molecule.append(
            [
                i_atom[0],
                i_atom[1][0],
                i_atom[1][1],
                i_atom[1][2],
            ]
        )

    dict_i_name = f"{dataset}-{items["curve"]}-{items["curve_x"]}"
    dict_[dict_i_name] = molecule
    dict_[f"molecule_{dataset}"].append(dict_i_name)
    dict_["molecule"].append(dict_i_name)
    dict_["charge"][dict_i_name] = mol.charge
    dict_["spin"][dict_i_name] = mol.spin

    if items["curve"] not in curve_geom_dict:
        curve_geom_dict[items["curve"]] = [molecule]
    else:
        curve_geom_dict[items["curve"]].append(molecule)

for key, value in curve_geom_dict.items():
    molecule1 = value[1]
    molecule0 = value[0]
    mol1 = []
    mol2 = []
    for i, atom in enumerate(molecule1):
        distance = np.linalg.norm(np.array(atom[1:]) - np.array(molecule0[i][1:]))
        if distance < 1e-5:
            mol1.append(atom)
        else:
            mol2.append(atom)

    dict_i_name = f"{dataset}-{key}-a"
    dict_[dict_i_name] = mol1
    dict_[f"molecule_{dataset}"].append(dict_i_name)
    dict_["molecule"].append(dict_i_name)
    dict_["charge"][dict_i_name] = 0
    dict_["spin"][dict_i_name] = 0

    dict_i_name = f"{dataset}-{key}-b"
    dict_[dict_i_name] = mol2
    dict_[f"molecule_{dataset}"].append(dict_i_name)
    dict_["molecule"].append(dict_i_name)
    dict_["charge"][dict_i_name] = 0
    dict_["spin"][dict_i_name] = 0


for items in s66x8["items"]:
    i_reaction = f"{items["curve"]}_{items["curve_x"]}"
    systems = [
        f"{dataset}-{items['curve']}-{items['curve_x']}",
        f"{dataset}-{items['curve']}-a",
        f"{dataset}-{items['curve']}-b",
    ]
    stoichiometry = [1, 1, -1]
    reference = items["reference_value"]
    dict_[f"reaction-{dataset}"][i_reaction] = {
        "systems": systems,
        "stoichiometry": stoichiometry,
        "reference": reference,
    }

In [4]:
with open(f"gmtkn-{dataset}-{BASIS}.json", "w") as f:
    json.dump(dict_, f)